# 🏈 Predictor NFL v3

Agente de predicción de juegos NFL sobre datos [nflverse](https://github.com/nflverse). Todo el pipeline vive en **`nfl_pred.py`** — este notebook es la interfaz interactiva.

**Predice por juego:** puntos por equipo, totales, spread, probabilidad de victoria calibrada.
**Props por jugador:** yardas (pase/tierra/recepción) con rango propio por jugador, **TDs** (esperado + prob de ≥1), **recepciones** e **intercepciones**.

**Modelo dual:**
- `modelo_vegas` (usa spread/total de Vegas) → marcadores finos
- `modelo_puro` (sin Vegas) → opinión propia; cuando discrepa >4 pts de Vegas se marca **`value`**

**Features:** forma últimos 5 juegos (puntos, yardas, EPA of/def), localía, descanso, juego divisional, clima (domo/temp/viento), líneas de Vegas.

**Automatización:** GitHub Actions corre `actualizar.py` cada martes y commitea `predicciones.csv`.

## 1. Inicializar (carga datos + entrena)

Primera vez descarga ~300 MB (después cachea). `walk_forward=True` agrega el backtest honesto (re-entrena semana a semana de 2025, ~1-2 min extra).

In [1]:
from nfl_pred import *

ctx = inicializar(walk_forward=True)

Aviso: reportes de lesion aun no publicados (salen con la temporada)
Juegos: 1693 | Jugador-semanas: 112447 | Jugadas pbp: 294989
Calendario 2026: 272 juegos | Roster: 2930 | Depth chart: 3172


Tabla equipo-juego: 3386 filas | matriz modelo: 3322 filas


--- Backtest 2025 (split simple) ---
MAE puntos por equipo: 7.49 | MAE total juego: 10.92
Acierto ganador - modelo puro: 61.8% | modelo c/Vegas: 61.1% | linea Vegas: 65.6% | prob calibrada: 66.3%


MAE yardas_pase: 72.49  (664 jugador-juegos)


MAE yardas_tierra: 17.23  (2309 jugador-juegos)


MAE yardas_recepcion: 16.27  (5481 jugador-juegos)
MAE td_pase: 0.92  (664 jugador-juegos)


MAE td_tierra: 0.32  (2309 jugador-juegos)


MAE td_recepcion: 0.23  (5481 jugador-juegos)


MAE recepciones: 1.25  (5481 jugador-juegos)
MAE intercepciones: 0.67  (664 jugador-juegos)


--- Backtest walk-forward 2025 (285 juegos, re-entrenando cada semana) ---
MAE puntos: 7.57 | acierto puro: 60.4% | c/Vegas: 63.5% | linea Vegas: 65.6%


**Cómo leer las métricas:**
- *split simple* = entrena 2020–2024, predice 2025 completo de una vez
- *walk-forward* = re-entrena cada semana con lo disponible hasta entonces (así funcionará en 2026 real)
- *prob calibrada* = regresión logística sobre `[diferencial del modelo puro, spread de Vegas]`; calibrada in-sample sobre 2025 — tómala como aproximación
- La línea de Vegas es el techo práctico (~66%); el valor del modelo está en las **discrepancias** y las **props**

## 2. Predecir un juego

Busca el juego en el calendario oficial 2026 (líneas de Vegas, descanso, clima/domo, división). Rosters y depth charts oficiales; excluye lesionados cuando hay reporte.

```python
predecir_juego('SEA', 'NE')          # visitante NE @ local SEA
predecir_juego('KC', 'BUF', week=5)  # semana específica
```

In [2]:
props = predecir_juego('SEA', 'NE')
props

Juego oficial: semana 1 (2026-09-09) | Vegas: spread +3.5, total 44.5
=== NE @ SEA ===
Marcador predicho:  SEA 24.6 - NE 20.0
Puntos totales:     44.6
Modelo puro (sin Vegas): SEA por +8.9 | discrepancia vs Vegas: +5.4  << posible value
Prob. victoria (calibrada): SEA 69% | NE 31%


  Sin historial NFL (SEA): Jadarian Price (RB)


  Sin historial NFL (NE): Eli Raridon (TE)


,equipo,jugador,pos,prop,prediccion,rango_68pct,prob_1mas
0,NE,Drake Maye,QB,intercepciones,0.42,NaN,0.34
1,NE,A.J. Brown,WR,recepciones,4.90,NaN,0.99
2,NE,Romeo Doubs,WR,recepciones,3.22,NaN,0.96
3,NE,Kayshon Boutte,WR,recepciones,2.97,NaN,0.95
4,NE,Hunter Henry,TE,recepciones,2.89,NaN,0.94
5,NE,Rhamondre Stevenson,RB,recepciones,2.61,NaN,0.93
6,NE,TreVeyon Henderson,RB,recepciones,1.13,NaN,0.68
7,NE,Drake Maye,QB,td_pase,1.21,NaN,0.70
8,NE,A.J. Brown,WR,td_recepcion,0.34,NaN,0.29
9,NE,Romeo Doubs,WR,td_recepcion,0.27,NaN,0.23


**Columnas de props:**
- `rango_68pct` — rango propio del jugador (quantile regression: jugadores estables → rango angosto)
- `prob_1mas` — probabilidad de al menos 1 TD / 1 intercepción (Poisson)

## 3. Predecir una semana completa

`discrepancia` = spread del modelo puro − spread de Vegas. `value=True` cuando |discrepancia| > 4 pts: juegos donde el modelo "ve algo" distinto al mercado.

In [3]:
semana = predecir_semana(1)
semana

,season,week,fecha,local,visitante,pred_local,pred_visita,total_pred,total_vegas,spread_pred,spread_puro,spread_vegas,discrepancia,value,ganador,prob
0,2026,1,2026-09-09,SEA,NE,24.6,20.0,44.6,44.5,4.6,8.9,3.5,5.4,True,SEA,0.69
1,2026,1,2026-09-10,LA,SF,25.4,23.8,49.2,48.5,1.7,2.7,3.0,-0.3,False,LA,0.60
2,2026,1,2026-09-13,CAR,CHI,21.9,21.3,43.2,44.5,0.5,-0.9,-2.5,1.6,False,CHI,0.60
3,2026,1,2026-09-13,CIN,TB,26.9,24.9,51.8,50.5,2.0,4.8,3.5,1.3,False,CIN,0.64
4,2026,1,2026-09-13,DET,NO,28.0,25.8,53.8,48.5,2.2,-10.1,7.0,-17.1,True,DET,0.55
5,2026,1,2026-09-13,HOU,BUF,23.5,24.2,47.7,45.5,-0.8,-0.5,-1.5,1.0,False,BUF,0.57
6,2026,1,2026-09-13,IND,BAL,21.0,27.7,48.8,49.5,-6.7,-14.0,-3.5,-10.5,True,BAL,0.77
7,2026,1,2026-09-13,JAX,CLE,27.7,18.6,46.3,40.5,9.0,12.6,7.5,5.1,True,JAX,0.81
8,2026,1,2026-09-13,PIT,ATL,21.8,18.3,40.1,42.5,3.5,3.6,3.0,0.6,False,PIT,0.61
9,2026,1,2026-09-13,TEN,NYJ,19.4,18.6,38.0,39.5,0.8,7.8,3.0,4.8,True,TEN,0.66


In [4]:
# solo los juegos con posible value
semana[semana['value']]

,season,week,fecha,local,visitante,pred_local,pred_visita,total_pred,total_vegas,spread_pred,spread_puro,spread_vegas,discrepancia,value,ganador,prob
0,2026,1,2026-09-09,SEA,NE,24.6,20.0,44.6,44.5,4.6,8.9,3.5,5.4,True,SEA,0.69
4,2026,1,2026-09-13,DET,NO,28.0,25.8,53.8,48.5,2.2,-10.1,7.0,-17.1,True,DET,0.55
6,2026,1,2026-09-13,IND,BAL,21.0,27.7,48.8,49.5,-6.7,-14.0,-3.5,-10.5,True,BAL,0.77
7,2026,1,2026-09-13,JAX,CLE,27.7,18.6,46.3,40.5,9.0,12.6,7.5,5.1,True,JAX,0.81
9,2026,1,2026-09-13,TEN,NYJ,19.4,18.6,38.0,39.5,0.8,7.8,3.0,4.8,True,TEN,0.66
10,2026,1,2026-09-13,LAC,ARI,24.1,23.3,47.5,45.5,0.8,-0.2,11.5,-11.7,True,LAC,0.78
11,2026,1,2026-09-13,LV,MIA,23.6,22.3,45.9,41.5,1.3,-3.8,3.0,-6.8,True,LV,0.52
12,2026,1,2026-09-13,MIN,GB,18.6,24.5,43.1,44.5,-5.9,9.0,-1.5,10.5,True,MIN,0.55
14,2026,1,2026-09-13,NYG,DAL,21.1,22.3,43.4,48.5,-1.1,5.1,-2.5,7.6,True,DAL,0.53


## 4. Registro de predicciones

- `guardar_predicciones(week)` — anexa a `predicciones.csv` (sin duplicar semanas)
- `evaluar_predicciones()` — acierto real contra resultados y contra Vegas

En temporada esto lo hace solo GitHub Actions cada martes; también puedes correrlo a mano.

In [5]:
guardar_predicciones(1)
evaluar_predicciones()

16 juegos de semana 1 guardados en predicciones.csv
Aun no hay resultados para las predicciones guardadas


## 5. Notas

**Flujo semanal automático (GitHub Actions):** martes 14:00 UTC corre `actualizar.py` → predice la semana próxima → commitea `predicciones.csv`. Disparo manual: pestaña *Actions* → *Predicciones semanales* → *Run workflow*.

**Flujo manual:** `nfl.clear_cache()` → re-ejecutar este notebook → `guardar_predicciones(week)`.

**Limitaciones honestas:**
- En offseason la "forma" viene de fin de 2025; los rosters ya son 2026 pero los números individuales son del equipo anterior del jugador. Se corrige solo al correr semanas de 2026 (el pipeline las incluye automáticamente cuando existen).
- Novatos sin historial NFL se omiten en props (se avisa).
- La probabilidad calibrada se ajustó sobre el mismo backtest 2025 (in-sample); re-calibrar tras unas semanas de 2026.